In [ ]:
import os
import sys
import pandas as pd
import numpy as np

from pathlib import Path
project_root = Path(os.path.dirname(os.getcwd()))
sys.path.append(str(project_root.joinpath('src')))

from utils.decoder_functions import decode_all, distance_all, example_decoder_weights
from utils.equivariance_functions import example_equiv_projs, equivariance_summary_all, equivariance_ccnorm_vs_dim_all
from utils.plotting_functions import plot_classification_errors, plot_orientation_errors, \
    plot_classification_errors_acrossstim, plot_orientation_errors_acrossstim, \
    plot_classification_errors_acrossrelori, plot_orientation_errors_acrossrelori, \
    plot_classification_errors_acrossC, plot_classification_errors_acrossN, plot_orientation_errors_acrossN, \
    plot_orientation_errors_acrossrefs, \
    plot_equivariance_dim, plot_equivariance_pcorr, plot_equivariance_ccnorm, plot_equivariance_variance, plot_equivariance_ccnorm_vs_dim, \
    plot_distance_all, plot_equivariance_distance_all, plot_relcenter_all, plot_equivariance_relcenter_all, \
    plot_example_decoder_weights

In [ ]:
model_df = pd.read_csv( project_root / 'params' / 'model_params.csv', header=0)
params_df = pd.read_csv( project_root / 'params' / 'fixed_params.csv', header=0, index_col=0)

In [ ]:
#colors for areas and conv layers
color_range = ['rgba(211, 211, 211, 1.0)', 'rgba(169, 169, 169, 1.0)', 'rgba(128, 128, 128, 1.0)', 'rgba(105, 105, 105, 1.0)', 'rgba(0, 0, 0, 1.0)']
color_dict = {'pixels': f'{color_range[0]}', 'V1': f'{color_range[1]}', 'LM': f'{color_range[2]}', \
              'RL': f'{color_range[3]}', 'AL': f'{color_range[4]}',\
              'conv1': f'{color_range[0]}', 'conv2': f'{color_range[1]}', 'conv3': f'{color_range[2]}', \
              'conv4': f'{color_range[3]}', 'conv5': f'{color_range[4]}'}

#colors for regularization parameters
color_range2 = ['rgba(211, 211, 211, 1.0)', 'rgba(169, 169, 169, 1.0)', 'rgba(128, 128, 128, 1.0)', 'rgba(105, 105, 105, 1.0)', 'rgba(0, 0, 0, 1.0)']
color_dict2 = {'0.0001': f'{color_range2[0]}', '0.001': f'{color_range2[1]}', '0.01': f'{color_range2[2]}', '0.1': f'{color_range2[3]}', '1': f'{color_range2[4]}'}
color_dict3 = {'0.001': f'{color_range2[0]}', '0.01': f'{color_range2[1]}', '0.1': f'{color_range2[2]}', '1': f'{color_range2[3]}', '10': f'{color_range2[4]}'}

### decoding across stimuli and relative orientations (ED Fig 2)
texture and orientation decoding sliced by stimulus and by relative orientation;
uses the default (250-neuron) decoding caches produced in plot_figures.ipynb

In [ ]:
#ED Fig 2a top
# figure stored in project_root / 'results_extended' / 'svm_stim_0.001C_250avg.pdf'
plot_classification_errors_acrossstim(model_df, params_df, project_root)
#ED Fig 2a bottom
# figure stored in project_root / 'results_extended' / 'ori_stim_0ref_250avg.pdf'
plot_orientation_errors_acrossstim(model_df, params_df, project_root)

#ED Fig 2b top/bottom
# figures stored in project_root / 'results_extended' / 'svm1_relori_.pdf' (top), 'svm2_relori_.pdf' (bottom)
plot_classification_errors_acrossrelori(model_df, params_df, project_root, color_dict)
#ED Fig 2c top/bottom
# figures stored in project_root / 'results_extended' / 'ori1_relori.pdf' (top), 'ori2_relori.pdf' (bottom)
plot_orientation_errors_acrossrelori(model_df, params_df, project_root, color_dict)

### decoding across regularization strength and neuron number (ED Fig 3)
takes a long time to run the first time

In [ ]:
#decode across regularization parameters
c_range = [0.0001, 0.001, 0.01, 0.1, 1]
decode_all(model_df, params_df, project_root, c_range=c_range, equivariant=False, orientation_decoding=False)

#ED Fig 3a
# figure stored in project_root / 'results_extended' / 'svm_Crange_250avg.pdf'
plot_classification_errors_acrossC(model_df, params_df, project_root, color_dict2)

In [ ]:
#decode across number of neurons
params_df_mod = params_df.copy()
n_neurons_list = list(np.arange(50, 1550, 50))
n_neurons_list.append(4000)

c_range = [0.001]
for n_neurons in n_neurons_list:
    params_df_mod.loc['n_neurons', 'value'] = n_neurons
    decode_all(model_df, params_df_mod, project_root, c_range=c_range, equivariant=False)

#ED Fig 3b top/bottom
# figures stored in project_root / 'results_extended' / 'svm1_0.001C_Navg.pdf' (top), 'svm2_0.001C_Navg.pdf' (bottom)
plot_classification_errors_acrossN(model_df, params_df, project_root, color_dict, C0=c_range[0], equivariant=False)
#ED Fig 3c top/bottom
# figures stored in project_root / 'results_extended' / 'ori1_Navg.pdf' (top), 'ori2_Navg.pdf' (bottom)
plot_orientation_errors_acrossN(model_df, params_df, project_root, color_dict, equivariant=False, ymin=0)

### all neurons: robustness of the main results (ED Fig 4)
takes a long time to run the first time

In [ ]:
#all-neuron params: same pipeline as the main analysis but with the full model list and a
#larger neuron budget (n_neurons=4000) under the default additive-gaussian noise model
model_df_all = pd.read_csv( project_root / 'params' / 'model_params_all.csv', header=0)
params_df_all = pd.read_csv( project_root / 'params' / 'fixed_params_all.csv', header=0, index_col=0)

In [ ]:
### first run takes a long time
# re-runs AlexNet inference and the equivariance / decoding analyses with n_neurons=4000
# under the default additive-gaussian noise model; every cache and figure gets a '4000avg'
# token (n_neurons=4000) so the default 250-neuron results are not overwritten.
# Results are cached and reused on subsequent runs.
example_equiv_projs(model_df_all, params_df_all, project_root)
equivariance_summary_all(model_df_all, params_df_all, project_root)
decode_all(model_df_all, params_df_all, project_root, c_range=[0.001])

In [ ]:
#ED Fig 4a top (Fig 1d analogue): classification (texture) decoding error
# figure stored in project_root / 'results' / 'svm_0.001C_4000avg.pdf'
plot_classification_errors(model_df_all, params_df_all, project_root, C0=0.001, symb='square', ymin=0.5)

#ED Fig 4a bottom (Fig 1e analogue): absolute orientation estimation error
# figure stored in project_root / 'results' / 'ori_0ref_4000avg.pdf'
plot_orientation_errors(model_df_all, params_df_all, project_root, symb='square')

#ED Fig 4b top (Fig 3b bottom analogue): dimension of the invariant and tuned subspaces
# figure stored in project_root / 'results' / 'dim_4000avg_20dim.pdf'
plot_equivariance_dim(model_df_all, params_df_all, project_root)

#ED Fig 4b bottom (Fig 4b analogue): Pearson correlation of the invariant and tuned subspaces
# figure stored in project_root / 'results' / 'pcorr_4000avg_20dim.pdf'
plot_equivariance_pcorr(model_df_all, params_df_all, project_root)

#ED Fig 4c top (ED Fig 10b analogue): normalized correlation coefficient (CCnorm)
# figure stored in project_root / 'results_extended' / 'cc_4000avg_20dim.pdf'
plot_equivariance_ccnorm(model_df_all, params_df_all, project_root, show_mice=False)

#ED Fig 4c bottom (ED Fig 10c analogue): variance explained by tuned and invariant dimensions
# figure stored in project_root / 'results_extended' / 'var_4000avg_20dim.pdf'
plot_equivariance_variance(model_df_all, params_df_all, project_root, show_mice=False)

### response distances (ED Fig 5a bottom, ED Fig 5b)

In [ ]:
#compute pairwise response distances and relative center distances (cached after first run)
distance_all(model_df, params_df, project_root)

#ED Fig 5a bottom
# figure stored in project_root / 'results_extended' / 'distances_avg.pdf'
plot_distance_all(model_df, params_df, project_root)
# 20-dim subspace version (exploratory; not a paper figure)
# figure stored in project_root / 'results_extended' / 'distances_avg_20dim.pdf'
plot_equivariance_distance_all(model_df, params_df, project_root)

#ED Fig 5b
# figure stored in project_root / 'results_extended' / 'relcenter_avg.pdf'
plot_relcenter_all(model_df, params_df, project_root, ymax=1.3)
# 20-dim subspace version (exploratory; not a paper figure)
# figure stored in project_root / 'results_extended' / 'relcenter_avg_20dim.pdf'
plot_equivariance_relcenter_all(model_df, params_df, project_root, ymax=10)

### normalized correlation coefficient and variance explained (ED Fig 10)

In [ ]:
#CCnorm as a function of the fitted subspace dimension d
#first run takes a long time; per-layer sweeps are cached and reused afterwards
#ED Fig 10a top/bottom
# figures stored in project_root / 'results_extended' / 'cc_vs_dim1_250avg_20dim.pdf' (top: mouse areas + pixels)
#                                                       'cc_vs_dim2_250avg_20dim.pdf' (bottom: AlexNet)
equivariance_ccnorm_vs_dim_all(model_df, params_df, project_root)
plot_equivariance_ccnorm_vs_dim(model_df, params_df, project_root, color_dict)

In [ ]:
#normalized correlation coefficient (CCnorm) of the tuned subspace
#ED Fig 10b
# figure stored in project_root / 'results_extended' / 'cc_250avg_20dim.pdf'
plot_equivariance_ccnorm(model_df, params_df, project_root, show_mice=False)

In [ ]:
#variance explained by tuned and invariant dimensions
#ED Fig 10c
# figure stored in project_root / 'results_extended' / 'var_250avg_20dim.pdf'
plot_equivariance_variance(model_df, params_df, project_root, show_mice=False)

### decoding across reference orientations (SI Fig 3)
takes a long time to run the first time

In [ ]:
#decode across different reference orientations
ref_index_list = range(15)
for ref_index in ref_index_list:
    print('Reference index: ', ref_index)
    decode_all(model_df, params_df, project_root, ref_index=ref_index, equivariant=False, class_decoding=False)
    decode_all(model_df, params_df, project_root, ref_index=ref_index, equivariant=True, class_decoding=False)

#SI Fig 3a top/bottom
# figures stored in project_root / 'results_extended' / 'ori1_refs.pdf' (top), 'ori2_refs.pdf' (bottom)
plot_orientation_errors_acrossrefs(model_df, params_df, project_root, color_dict)
#SI Fig 3b top/bottom
# figures stored in project_root / 'results_extended' / 'ori1_refs_20dim.pdf' (top), 'ori2_refs_20dim.pdf' (bottom)
plot_orientation_errors_acrossrefs(model_df, params_df, project_root, color_dict, equivariant=True)

### equivariant decoding across regularization strength (SI Fig 4)

In [ ]:
#decode across regularization parameters
c_range = [0.001, 0.01, 0.1, 1, 10]
decode_all(model_df, params_df, project_root, c_range=c_range, equivariant=True, orientation_decoding=False)

#SI Fig 4
# figure stored in project_root / 'results_extended' / 'svm_Crange_250avg_20dim.pdf'
plot_classification_errors_acrossC(model_df, params_df, project_root, color_dict3, equivariant=True)

### exploratory analyses (not in the paper)
sanity checks and example weights that are not used as paper-figure panels
(the gain-model example decoder weights of ED Fig 6 are produced in gain_model_analyses.ipynb)

In [ ]:
#control: decode from the fitted rotation-model predictions instead of the recorded
#responses, in the invariant/tuned subspaces (exploratory; not a paper figure)
c_range = [1]
decode_all(model_df, params_df, project_root, c_range=c_range, equivariant=True, model=True)

plot_classification_errors(model_df, params_df, project_root, C0=c_range[0], equivariant=True, model=True)
plot_orientation_errors(model_df, params_df, project_root, equivariant=True, model=True, ymin=0)

In [ ]:
#example decoder weights for the full-space decoder
example_decoder_weights(model_df, params_df, project_root, C0=0.001)

# example decoder weights for all models (exploratory; not a paper figure)
# figures stored in project_root / 'results_extended' / 'example_weights' / '{model}_{layer}_example_decoder_250rand_comp_*.pdf'
# (ED Fig 6 in the paper is the gain-model version produced in gain_model_analyses.ipynb)
plot_example_decoder_weights(model_df, params_df, project_root)

In [ ]:
#example decoder weights for the subspace decoder (exploratory; not a paper figure)
example_decoder_weights(model_df, params_df, project_root, C0=0.001, equivariant=True)

plot_example_decoder_weights(model_df, params_df, project_root, equivariant=True)